<a href="https://colab.research.google.com/github/jairaj023/Internship-Practice/blob/main/Day_8Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
import spacy
import pandas as pd
from spacy.matcher import PhraseMatcher

In [3]:
from google.colab import files
upload = files.upload()

Saving raw_jobs.csv to raw_jobs.csv


In [5]:
df = pd.read_csv("raw_jobs.csv")

In [7]:
print(df.head())

     job_id         job_title  company              location  \
0  JOB00001  Python Developer  Infosys                Remote   
1  JOB00002    Data Scientist  Infosys  Hyderabad, Telangana   
2  JOB00003     Data Engineer      IBM  Bengaluru, Karnataka   
3  JOB00004  Business Analyst   Google  Hyderabad, Telangana   
4  JOB00005  Business Analyst  Mphasis          Delhi, India   

                                     job_description  experience education  \
0  We are looking for a Python Developer to join ...   1-3 years       MCA   
1  We are looking for a Data Scientist to join ou...   0-2 years       MCA   
2  We are looking for a Data Engineer to join our...  8-12 years       BCA   
3  We are looking for a Business Analyst to join ...   1-3 years    M.Tech   
4  We are looking for a Business Analyst to join ...  8-12 years    M.Tech   

       salary    job_type  
0  ₹10-16 LPA   Part-time  
1  ₹12-20 LPA      Remote  
2    ₹3-5 LPA  Internship  
3  ₹10-16 LPA      Remote  
4    ₹

In [8]:
print(df.columns)

Index(['job_id', 'job_title', 'company', 'location', 'job_description',
       'experience', 'education', 'salary', 'job_type'],
      dtype='object')


In [9]:
text_column = "job_description"

In [10]:
print(df[text_column].head())

0    We are looking for a Python Developer to join ...
1    We are looking for a Data Scientist to join ou...
2    We are looking for a Data Engineer to join our...
3    We are looking for a Business Analyst to join ...
4    We are looking for a Business Analyst to join ...
Name: job_description, dtype: object


In [11]:
regex_patterns = {
    "Python": r"\bpython(?:\s*3)?(?:\s+programming)?\b",
    "Machine Learning": r"\bmachine\s+learning\b",
    "Deep Learning": r"\bdeep\s+learning\b",
    "Natural Language Processing": r"\bnatural\s+language\s+processing\b",
    "Power BI": r"\bpower\s*bi\b",
    "Data Science": r"\bdata\s+science\b"
}

In [12]:
def regex_extract_skills(text):

    if pd.isna(text):
        return []

    text = str(text)

    found_skills = []

    for skill, pattern in regex_patterns.items():

        if re.search(pattern, text, re.IGNORECASE):
            found_skills.append(skill)

    return found_skills

In [13]:
df["regex_skills"] = df[text_column].apply(regex_extract_skills)

In [14]:
print(df[[text_column, "regex_skills"]].head())

                                     job_description  \
0  We are looking for a Python Developer to join ...   
1  We are looking for a Data Scientist to join ou...   
2  We are looking for a Data Engineer to join our...   
3  We are looking for a Business Analyst to join ...   
4  We are looking for a Business Analyst to join ...   

                 regex_skills  
0                    [Python]  
1  [Python, Machine Learning]  
2                    [Python]  
3                  [Power BI]  
4                  [Power BI]  


In [15]:
nlp = spacy.load("en_core_web_sm")

In [16]:
skills = [
    "Python",
    "Machine Learning",
    "Deep Learning",
    "Natural Language Processing",
    "Power BI",
    "Data Science"
]

In [17]:
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

patterns = [nlp.make_doc(skill) for skill in skills]

matcher.add("SKILLS", patterns)

In [18]:
def spacy_extract_skills(text):

    if pd.isna(text):
        return []

    doc = nlp(str(text))

    matches = matcher(doc)

    found_skills = []

    for match_id, start, end in matches:

        skill = doc[start:end].text

        for standard_skill in skills:

            if skill.lower() == standard_skill.lower():
                found_skills.append(standard_skill)

    return list(set(found_skills))

In [19]:
df["spacy_skills"] = df[text_column].apply(spacy_extract_skills)

In [20]:
print(df[[text_column, "spacy_skills"]].head())

                                     job_description  \
0  We are looking for a Python Developer to join ...   
1  We are looking for a Data Scientist to join ou...   
2  We are looking for a Data Engineer to join our...   
3  We are looking for a Business Analyst to join ...   
4  We are looking for a Business Analyst to join ...   

                 spacy_skills  
0                    [Python]  
1  [Python, Machine Learning]  
2                    [Python]  
3                  [Power BI]  
4                  [Power BI]  


In [21]:
def combine_skills(row):

    skills = row["regex_skills"] + row["spacy_skills"]

    return list(set(skills))

In [22]:
df["final_skills"] = df.apply(combine_skills, axis=1)

In [23]:
print(df[[text_column, "final_skills"]].head(10))

                                     job_description  \
0  We are looking for a Python Developer to join ...   
1  We are looking for a Data Scientist to join ou...   
2  We are looking for a Data Engineer to join our...   
3  We are looking for a Business Analyst to join ...   
4  We are looking for a Business Analyst to join ...   
5  We are looking for a Software Engineer to join...   
6  We are looking for a Frontend Developer to joi...   
7  We are looking for a AI Engineer to join our t...   
8  We are looking for a Frontend Developer to joi...   
9  We are looking for a AI Engineer to join our t...   

                                final_skills  
0                                   [Python]  
1                 [Python, Machine Learning]  
2                                   [Python]  
3                                 [Power BI]  
4                                 [Power BI]  
5                                   [Python]  
6                                         []  
7      

In [24]:
df["final_skills"] = df["final_skills"].apply(lambda x: ", ".join(x))